# Flatten Nested JSON — PySpark Utility

This notebook provides reusable PySpark functions to flatten deeply nested JSON structures, including:
- Recursively flattening `StructType` columns (joining field names with `.`)
- Exploding `ArrayType` columns into individual rows

Used in downstream Silver layer processing after raw JSON is ingested into the Bronze layer.

## Import necessary packages

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, ArrayType

## Function to flatten nested JSON

### `flatten_structs(nested_df)`
Recursively flattens all `StructType` columns in a DataFrame. Nested field names are joined using `.` as a separator.

### `flatten_df(df, arrays_to_not_explode=[])`
Flattens structs AND explodes all `ArrayType` columns. Pass column names to `arrays_to_not_explode` to skip exploding specific arrays.

In [ ]:
# Flatten structs
def flatten_structs(nested_df):
    """
    Recursively flattens all StructType columns in a PySpark DataFrame.
    Nested field names are joined with '.' separator.

    Args:
        nested_df (DataFrame): Input DataFrame with nested StructType columns.

    Returns:
        DataFrame: Flattened DataFrame.
    """
    stack = [((,), nested_df)]
    columns = []

    while len(stack) > 0:
        parents, df = stack.pop()

        flat_cols = [
            col(".".join(parents + (c[0],))).alias(".".join(parents + (c[0],)))
            for c in df.dtypes
            if c[1][:6] != "struct"
        ]

        nested_cols = [
            c[0]
            for c in df.dtypes
            if c[1][:6] == "struct"
        ]

        columns.extend(flat_cols)

        for nested_col in nested_cols:
            projected_df = df.select(nested_col + ".*")
            stack.append((parents + (nested_col,), projected_df))

    return nested_df.select(columns)


# Flatten arrays
def flatten_df(df, arrays_to_not_explode=[]):
    """
    Flattens all StructType columns and explodes all ArrayType columns in a PySpark DataFrame.
    Columns listed in arrays_to_not_explode will not be exploded.

    Args:
        df (DataFrame): Input DataFrame.
        arrays_to_not_explode (list): List of column names to skip when exploding arrays.

    Returns:
        DataFrame: Fully flattened DataFrame.
    """
    def is_valid_array(array_col):
        element_type = df.schema[array_col].dataType.elementType
        if isinstance(element_type, StructType) or isinstance(element_type, ArrayType):
            return True
        else:
            return False

    df = flatten_structs(df)

    array_cols = [
        c[0]
        for c in df.dtypes
        if c[1][:5] == "array"
    ]

    while len(array_cols) > 0:
        for c in array_cols:
            if is_valid_array(c):
                df = df.withColumn(c, explode_outer(c))
            else:
                arrays_to_not_explode.append(c)
        df = flatten_structs(df)
        array_cols = [c[0] for c in df.dtypes if c[0] not in arrays_to_not_explode and c[1][:5] == "array"]

    return df

## Example Usage

The following example demonstrates flattening a nested JSON structure representing restaurant bill data — with nested customer info, order details, and an array of items.

### Input Schema
```
root
 └── data: struct
      ├── customer_name: string
      ├── outlet: string
      ├── order_details: struct
      │    ├── total_amount: double
      │    ├── tax: double
      │    └── items: array
      │         ├── item_name: string
      │         ├── quantity: integer
      │         └── unit_price: double
      └── payment: struct
           ├── method: string
           └── status: string
```

### After `flatten_df()`
Each item in the `items` array becomes its own row, and all nested fields are promoted to top-level columns.

In [ ]:
# Example: Flatten nested bill data
from pyspark.sql import Row

# Sample data mimicking a Posist API response
sample_data = [
    Row(
        data=Row(
            customer_name="Alice",
            outlet="haridwar_food_court",
            order_details=Row(
                total_amount=450.0,
                tax=45.0,
                items=[
                    Row(item_name="Burger", quantity=2, unit_price=100.0),
                    Row(item_name="Fries", quantity=1, unit_price=50.0),
                    Row(item_name="Cola", quantity=2, unit_price=75.0)
                ]
            ),
            payment=Row(method="UPI", status="paid")
        )
    ),
    Row(
        data=Row(
            customer_name="Bob",
            outlet="haridwar_food_court",
            order_details=Row(
                total_amount=200.0,
                tax=20.0,
                items=[
                    Row(item_name="Pizza", quantity=1, unit_price=180.0),
                    Row(item_name="Water", quantity=1, unit_price=20.0)
                ]
            ),
            payment=Row(method="Cash", status="paid")
        )
    )
]

# Create DataFrame
nested_df = spark.createDataFrame(sample_data)

print("=== Original Schema ===")
nested_df.printSchema()
nested_df.show(truncate=False)

In [ ]:
# Apply flatten_df — flattens structs and explodes items array
flat_df = flatten_df(nested_df)

print("=== Flattened Schema ===")
flat_df.printSchema()

print("=== Flattened Data (each item is now a separate row) ===")
flat_df.show(truncate=False)

### Expected Output

After flattening, each item in the `items` array becomes its own row:

| data.customer_name | data.outlet | data.order_details.total_amount | data.order_details.tax | data.order_details.items.item_name | data.order_details.items.quantity | data.order_details.items.unit_price | data.payment.method | data.payment.status |
|---|---|---|---|---|---|---|---|---|
| Alice | haridwar_food_court | 450.0 | 45.0 | Burger | 2 | 100.0 | UPI | paid |
| Alice | haridwar_food_court | 450.0 | 45.0 | Fries | 1 | 50.0 | UPI | paid |
| Alice | haridwar_food_court | 450.0 | 45.0 | Cola | 2 | 75.0 | UPI | paid |
| Bob | haridwar_food_court | 200.0 | 20.0 | Pizza | 1 | 180.0 | Cash | paid |
| Bob | haridwar_food_court | 200.0 | 20.0 | Water | 1 | 20.0 | Cash | paid |

Note: Alice's single bill row has been expanded into 3 rows (one per item), and Bob's into 2 rows.

In [ ]:
# Tip: To skip exploding a specific array column, pass it to arrays_to_not_explode
# Example: keep 'data.order_details.items' as an array and don't explode it
flat_df_no_explode = flatten_df(nested_df, arrays_to_not_explode=["data.order_details.items"])

print("=== Flattened without exploding items ===")
flat_df_no_explode.printSchema()
flat_df_no_explode.show(truncate=False)